---
# Experiment 5: RNN — SMS Spam Detection
---

## 5.1 Aim
To build a **Recurrent Neural Network (RNN)** with an **LSTM** layer using TensorFlow/Keras to classify SMS messages as **spam** or **ham (legitimate)** using the UCI SMS Spam Collection dataset.

## 5.2 Required Libraries

In [4]:
!pip install tensorflow pandas numpy matplotlib scikit-learn

## 5.3 Import Libraries

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re, io, zipfile, requests

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
tf.random.set_seed(42)
print('Libraries imported.')

Libraries imported.


## 5.4 Dataset Description
The **SMS Spam Collection** dataset from UCI contains **5,574 SMS messages** labelled as:
- **ham** (4,827 messages — legitimate)
- **spam** (747 messages)

Each message is raw text. This is a classic binary text classification problem.

## 5.5 Data Loading

In [6]:
# Download the SMS Spam Collection dataset (UCI)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
with z.open('SMSSpamCollection') as f:
    df = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])

print(f'Dataset shape : {df.shape}')
print(f'Label counts  :\n{df["label"].value_counts()}')
df.head(10)

Dataset shape : (5572, 2)
Label counts  :
label
ham     4825
spam     747
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


## 5.6 Data Preprocessing

In [7]:
# Encode labels: ham=0, spam=1
df['label_enc'] = (df['label'] == 'spam').astype(int)

# Basic text cleaning
def clean_text(text):
    text = text.lower()                       # Lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

df['clean_msg'] = df['message'].apply(clean_text)
print('Sample cleaned messages:')
print(df[['message', 'clean_msg', 'label_enc']].head(4))

Sample cleaned messages:
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   

                                           clean_msg  label_enc  
0  go until jurong point crazy available only in ...          0  
1                            ok lar joking wif u oni          0  
2  free entry in 2 a wkly comp to win fa cup fina...          1  
3        u dun say so early hor u c already then say          0  


## 5.7 Tokenisation & Sequence Padding

In [8]:
MAX_VOCAB  = 10000   # Vocabulary size
MAX_LEN    = 100     # Pad/truncate all sequences to this length

# Tokenise text
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_msg'])

sequences = tokenizer.texts_to_sequences(df['clean_msg'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = df['label_enc'].values

print(f'Vocabulary size : {len(tokenizer.word_index)}')
print(f'X shape         : {X.shape}')
print(f'Sample sequence : {X[0][:20]} ...')

Vocabulary size : 9578
X shape         : (5572, 100)
Sample sequence : [  47  445 4362  796  714  682   65   10 1249   90  121  362 1250  155
 2882 1251   69   59 4363  138] ...


## 5.8 Train–Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (4457, 100), Test: (1115, 100)


## 5.9 Model Architecture — Bidirectional LSTM
- **Embedding Layer** : Learns dense word representations (dim=64)
- **Bidirectional LSTM** : Captures context from both directions (64 units)
- **Dense(32, ReLU)** + **Dropout(0.4)**
- **Dense(1, Sigmoid)** : Binary output (spam probability)

In [10]:
EMBED_DIM = 64
LSTM_UNITS = 64

model = keras.Sequential([
    layers.Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN, name='embedding'),
    layers.Bidirectional(
        layers.LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2),
        name='bilstm'
    ),
    layers.Dense(32, activation='relu', name='dense1'),
    layers.Dropout(0.4, name='dropout'),
    layers.Dense(1, activation='sigmoid', name='output')
], name='RNN_SpamDetector')

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "RNN_SpamDetector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 5.10 Compilation & Training

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/8
63/63 ━━━━━━━━━━━━━━━━━━━━ 38s 268ms/step - accuracy: 0.8851 - loss: 0.3211 - val_accuracy: 0.9462 - val_loss: 0.1754
Epoch 2/8
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.9698 - loss: 0.1172

## 5.11 Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc * 100:.2f}%')

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

## 5.12 Visualisation — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes,
                               [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
                               ['Accuracy', 'Loss']):
    ax.plot(history.history[metric[0]], label='Train', linewidth=2)
    ax.plot(history.history[metric[1]], label='Val',   linewidth=2, linestyle='--')
    ax.set_title(f'RNN LSTM — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5.13 Visualisation — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Ham', 'Spam'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Reds', colorbar=False)
ax.set_title('RNN — Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.14 Testing with New Sample Input

In [ ]:
def predict_sms(message):
    cleaned = clean_text(message)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    prob = model.predict(padded, verbose=0)[0][0]
    label = 'SPAM' if prob >= 0.5 else 'HAM'
    print(f'Message : "{message}"')
    print(f'Prediction : {label}  (confidence: {prob:.4f})')
    print()

predict_sms('Congratulations! You have won a free iPhone. Click here to claim now!')
predict_sms('Hey, are you coming to the meeting at 3pm today?')
predict_sms('URGENT! Your bank account has been suspended. Call 08001234 immediately.')

## 5.15 Observations & Conclusion
- The Bidirectional LSTM captures context from both left and right of each word, crucial for short SMS text.
- The model achieves high precision and recall for spam detection.
- The Embedding layer learns meaningful word representations from raw text during training.
- Spam messages typically contain urgent language, monetary promises, and unusual capitalisation.

## 5.16 Improvements & Future Work
- Use **pre-trained embeddings** (GloVe, Word2Vec) for better word representations.
- Try **1D CNN** or **Transformer-based** models for faster training.
- Handle class imbalance with **class_weight** in `model.fit()`.